# TrackPrior AI — Exploratory Data Analysis & Quality Audit

**Project:** TrackPrior AI — Railway Track Defect Severity Classification and Maintenance Prioritization System  
**Author:** Senior Data Engineer, ML Engineer & Railway Infrastructure Researcher  
**Date:** 2026-11-18  

This notebook walks through the collected public datasets, inspects data quality distributions, demonstrates the experimental data splits, and evaluates engineering threshold compliance.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visual styling
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = 'Arial'

BASE_DIR = os.path.dirname(os.path.abspath(''))
DATA_DIR = os.path.join(BASE_DIR, 'data')
print(f'TrackPrior AI Project Base: {BASE_DIR}')

## 1. Dataset Manifest Overview
Inspect the verified dataset manifest generated during automated collection and validation.

In [ ]:
manifest_file = os.path.join(DATA_DIR, 'manifests', 'dataset_manifest.json')
with open(manifest_file, 'r', encoding='utf-8') as f:
    manifest = json.load(f)

summary_rows = []
for ds in manifest['datasets']:
    summary_rows.append({
        'ID': ds['dataset_id'],
        'Name': ds['name'],
        'Category': ds['category'],
        'Records': f"{ds['records']:,}",
        'Coverage Period': ds['observation_period'],
        'Validation': ds['validation_status']
    })

pd.DataFrame(summary_rows)

## 2. Experiment A: Real Track Geometry Inspection Data (INFORMS 2015)
Inspect the cleaned observational geometry dataset, defect priority tags (`DEF_PRTY`), and defect types.

In [ ]:
informs_path = os.path.join(DATA_DIR, 'processed', 'informs_cleaned_geometry.parquet')
df_inf = pd.read_parquet(informs_path)
print(f'Total observations: {len(df_inf):,} rows | Columns: {len(df_inf.columns)}')
display(df_inf.head(5))

print('\nDefect Severity Class Balance:')
print(df_inf['defect_priority_tag'].value_counts(normalize=True).map(lambda n: f'{n*100:.2f}%'))

In [ ]:
# Chronological train and test split verification
train_a = pd.read_parquet(os.path.join(DATA_DIR, 'processed', 'exp_a_real_train.parquet'))
test_a = pd.read_parquet(os.path.join(DATA_DIR, 'processed', 'exp_a_real_test.parquet'))

print(f'Train split ({train_a.shape[0]:,} rows): {train_a["inspection_date"].min().date()} to {train_a["inspection_date"].max().date()}')
print(f'Test split  ({test_a.shape[0]:,} rows): {test_a["inspection_date"].min().date()} to {test_a["inspection_date"].max().date()}')
assert train_a['inspection_date'].max() < test_a['inspection_date'].min(), 'Chronological leakage detected!'

## 3. Experiment B: Synthetic Maintenance Prototype (Hugging Face)
Inspect the synthetic dataset, evaluate class imbalance, and verify the isolation of target-leakage features.

In [ ]:
synth_path = os.path.join(DATA_DIR, 'synthetic', 'synthetic_cleaned_track_subset.parquet')
df_synth = pd.read_parquet(synth_path)
print(f'Track-focused synthetic observations: {len(df_synth):,} rows')

print('\nOrdinal Severity Breakdown (0: None, 1: Low, 2: Medium, 3: High, 4: Critical):')
display(df_synth['severity_level_ordinal'].value_counts().sort_index())

## 4. Indian Railways Track Inspection Standards (IRPWM 2020)
Review verified engineering limits governing Indian Railways Broad Gauge track maintenance.

In [ ]:
std_path = os.path.join(DATA_DIR, 'raw', 'standards', 'irpwm_geometry_thresholds.csv')
df_std = pd.read_csv(std_path)
display(df_std[['parameter_name', 'threshold_value', 'measurement_unit', 'applicable_speed_or_track_category', 'maintenance_classification', 'page_number']].head(10))

## 5. RTI TMS Pipeline Verification
Demonstrate how the RTI pipeline (`scripts/rti_pipeline.py`) seamlessly normalizes incoming Zonal Railway TMS records.

In [ ]:
from rti_pipeline import RTIIndianRailwaysTMSPipeline

pipeline = RTIIndianRailwaysTMSPipeline()
sample_tms_record = pd.DataFrame({
    'rly': ['NR', 'WR', 'CR'],
    'division': ['Delhi', 'Mumbai Central', 'Nagpur'],
    'sec_name': ['NDLS-GZB', 'BCT-BVI', 'NGP-WR'],
    'track_no': ['UP Main', 'DN Local', 'Single Line'],
    'from_km': [14.2, 22.8, 45.1],
    'date_of_run': ['2024-05-10', '2024-05-11', '2024-05-12'],
    'param_name': ['Track Twist', 'Track Gauge (Slack)', 'Alignment Peak'],
    'amplitude': [4.5, 12.0, 4.0],
    'urgency': ['UML', 'UML', 'NBML']
})

mapped, missing = pipeline.map_columns(sample_tms_record)
std_sample = pipeline.standardize_severity(mapped)
display(std_sample[['zone', 'division', 'block_section', 'geometry_parameter', 'measured_value', 'standardized_severity']])